In [ ]:
# ============================================================
# NOTEBOOK: 01_data_download.ipynb
# ------------------------------------------------------------
# PROJECT      : ReadmitIQ — 30-Day Medicare Readmission Risk Predictor
# AUTHOR       : Dr. Nikki
# CREATED      : 2026-04-08
# LAST UPDATED : 2026-04-08
#
# PURPOSE
# -------
# This notebook is the entry point for the ReadmitIQ data pipeline.
# It guides the user through downloading the CMS DE-SynPUF Sample 1
# files, extracting them into the correct directory, validating that
# each file loaded correctly, and saving lightweight preview files
# for use in downstream notebooks.
#
# DATA SOURCE
# -----------
# CMS Linkable 2008-2010 Medicare Data Entrepreneurs Synthetic
# Public Use File (DE-SynPUF), Sample 1 of 20.
# Free download, no Data Use Agreement required.
# URL: https://www.cms.gov/data-research/statistics-trends-and-reports/medicare-claims-synthetic-public-use-files/cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf/de10-sample-1
#
# INPUTS
# ------
# Files must be manually downloaded from CMS (instructions below).
#
# OUTPUTS
# -------
# data/raw/
#   DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
#   DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv
#   DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv
#   DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv
#
# data/processed/
#   01_beneficiary_2008_preview.csv   (first 1000 rows)
#   01_inpatient_preview.csv          (first 1000 rows)
#   01_validation_report.txt          (row counts, dtypes, null summary)
#
# RUN ORDER
# ---------
# This is the FIRST notebook in the pipeline.
# Run before: 02_eda.ipynb
# ============================================================

## Section 0 — Download Instructions

Before running any code cells, you need to manually download the SynPUF files from CMS.
Follow these steps exactly:

**Step 1:** Go to this URL:  
https://www.cms.gov/data-research/statistics-trends-and-reports/medicare-claims-synthetic-public-use-files/cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf/de10-sample-1

**Step 2:** Download these four ZIP files (you only need Sample 1):
- `DE1.0 Sample 1 2008 Beneficiary Summary File (ZIP)`
- `DE1.0 Sample 1 2009 Beneficiary Summary File (ZIP)`
- `DE1.0 Sample 1 2010 Beneficiary Summary File (ZIP)`
- `DE1.0 Sample 1 2008-2010 Inpatient Claims (ZIP)`

**Step 3:** Extract each ZIP file into the `data/raw/` folder of this project.

**Step 4:** Confirm your `data/raw/` folder contains these four CSV files:
- `DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv`
- `DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv`
- `DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv`
- `DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv`

**Step 5:** Run the cells below in order.

> **Note:** These files are synthetic (not real patient data) and are in the public domain.
> They are excluded from version control via `.gitignore` — never commit raw data to GitHub.

## Section 1 — Imports and Configuration

In [1]:
# ---------------------------------------------------------
# Standard library and third-party imports
# ---------------------------------------------------------
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print(f"Python version  : {sys.version}")
print(f"Pandas version  : {pd.__version__}")
print(f"NumPy version   : {np.__version__}")
print(f"Notebook run at : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Python version  : 3.12.1 (main, Nov 27 2025, 10:47:52) [GCC 13.3.0]
Pandas version  : 3.0.2
NumPy version   : 2.4.4
Notebook run at : 2026-04-09 03:08:52


In [5]:
# ---------------------------------------------------------
# Directory configuration
# All paths are relative to the project root so this
# notebook works regardless of where the repo is cloned.
# ---------------------------------------------------------

# Navigate up one level from notebooks/ to reach the project root
PROJECT_ROOT = Path(os.getcwd()).parent

# Define key directories
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create directories if they do not already exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root  : {PROJECT_ROOT}")
print(f"Raw data dir  : {RAW_DIR}")
print(f"Processed dir : {PROCESSED_DIR}")

Project root  : /workspaces/readmitiq
Raw data dir  : /workspaces/readmitiq/data/raw
Processed dir : /workspaces/readmitiq/data/processed


In [6]:
# ---------------------------------------------------------
# Define the expected filenames for SynPUF Sample 1
# These are the exact filenames CMS uses inside the ZIPs.
# If CMS ever changes their naming convention, update here.
# ---------------------------------------------------------

FILES = {
    "beneficiary_2008": "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv",
    "beneficiary_2009": "DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv",
    "beneficiary_2010": "DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv",
    "inpatient"       : "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv",
}

print("Expected files:")
for key, filename in FILES.items():
    print(f"  [{key}] {filename}")

Expected files:
  [beneficiary_2008] DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
  [beneficiary_2009] DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv
  [beneficiary_2010] DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv
  [inpatient] DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv


## Section 2 — File Validation Check

Before we try to load anything, confirm all required files are present.
If any file is missing, the cell will tell you exactly which one and stop.

In [7]:
# ---------------------------------------------------------
# Check that every expected file exists in data/raw/
# This cell will print a clear pass/fail for each file
# and raise an error if anything is missing.
# ---------------------------------------------------------

print("Checking for required files in data/raw/ ...\n")

all_present = True

for key, filename in FILES.items():
    filepath = RAW_DIR / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / (1024 * 1024)
        print(f"  ✅ FOUND    [{key}]  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ MISSING  [{key}]  → Expected at: {filepath}")
        all_present = False

print()

if not all_present:
    raise FileNotFoundError(
        "One or more required files are missing from data/raw/. "
        "Please follow the download instructions in Section 0 before continuing."
    )

print("All required files are present. Proceeding to load.")

Checking for required files in data/raw/ ...

  ✅ FOUND    [beneficiary_2008]  (13.9 MB)
  ✅ FOUND    [beneficiary_2009]  (13.8 MB)
  ✅ FOUND    [beneficiary_2010]  (13.4 MB)
  ✅ FOUND    [inpatient]  (15.9 MB)

All required files are present. Proceeding to load.


## Section 3 — Load the Beneficiary Summary Files

The Beneficiary Summary file contains one row per synthetic Medicare beneficiary per year.
It holds demographics (age, sex, race) and chronic condition flags (diabetes, CHF, COPD, etc.).
We load all three years separately, then stack them into a single longitudinal file.

In [8]:
# ---------------------------------------------------------
# Load the 2008 Beneficiary Summary file
# This is the primary year — we anchor demographics here
# because the model's training cohort is based on 2008 data.
# ---------------------------------------------------------

print("Loading 2008 Beneficiary Summary...")

bene_2008 = pd.read_csv(
    RAW_DIR / FILES["beneficiary_2008"],
    low_memory=False    # Suppress dtype inference warnings on large files
)

# Tag each row with its source year so we can tell years apart after stacking
bene_2008["YEAR"] = 2008

print(f"  Rows    : {bene_2008.shape[0]:,}")
print(f"  Columns : {bene_2008.shape[1]}")
print(f"  Memory  : {bene_2008.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Loading 2008 Beneficiary Summary...
  Rows    : 116,352
  Columns : 33
  Memory  : 40.3 MB


In [9]:
# ---------------------------------------------------------
# Load the 2009 and 2010 Beneficiary Summary files
# Same structure as 2008 — just different years.
# We'll use 2009 as our test year in the modeling notebook.
# ---------------------------------------------------------

print("Loading 2009 Beneficiary Summary...")
bene_2009 = pd.read_csv(RAW_DIR / FILES["beneficiary_2009"], low_memory=False)
bene_2009["YEAR"] = 2009
print(f"  Rows: {bene_2009.shape[0]:,}  |  Cols: {bene_2009.shape[1]}")

print("\nLoading 2010 Beneficiary Summary...")
bene_2010 = pd.read_csv(RAW_DIR / FILES["beneficiary_2010"], low_memory=False)
bene_2010["YEAR"] = 2010
print(f"  Rows: {bene_2010.shape[0]:,}  |  Cols: {bene_2010.shape[1]}")

Loading 2009 Beneficiary Summary...
  Rows: 114,538  |  Cols: 33

Loading 2010 Beneficiary Summary...
  Rows: 112,754  |  Cols: 33


In [10]:
# ---------------------------------------------------------
# Stack all three beneficiary years into one dataframe.
# ignore_index=True resets the row index so there are no
# duplicate index values after stacking.
# ---------------------------------------------------------

print("Stacking all three beneficiary years...")

bene_all = pd.concat(
    [bene_2008, bene_2009, bene_2010],
    axis=0,
    ignore_index=True
)

print(f"  Combined rows : {bene_all.shape[0]:,}")
print(f"  Columns       : {bene_all.shape[1]}")
print(f"  Years present : {sorted(bene_all['YEAR'].unique())}")

# Quick look at the first few rows
print("\nFirst 3 rows of combined beneficiary data:")
bene_all.head(3)

Stacking all three beneficiary years...
  Combined rows : 343,644
  Columns       : 33
  Years present : [np.int64(2008), np.int64(2009), np.int64(2010)]

First 3 rows of combined beneficiary data:


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,...,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR,YEAR
0,00013D2EFD8E45D1,19230501,NaN,1,1,0,26,950,12,12,...,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0,2008
1,00016F745862898F,19430101,NaN,1,1,0,39,230,12,12,...,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0,2008
2,0001FDD721E223DC,19360901,NaN,2,1,0,39,280,12,12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2008


In [11]:
# ---------------------------------------------------------
# Inspect column names in the beneficiary file.
# This is important — we need to confirm that the chronic
# condition flag columns (SP_*) and the key identifier
# (DESYNPUF_ID) are present.
# ---------------------------------------------------------

print("All columns in the Beneficiary Summary file:\n")

for i, col in enumerate(bene_all.columns):
    print(f"  {i+1:>3}. {col}")

All columns in the Beneficiary Summary file:

    1. DESYNPUF_ID
    2. BENE_BIRTH_DT
    3. BENE_DEATH_DT
    4. BENE_SEX_IDENT_CD
    5. BENE_RACE_CD
    6. BENE_ESRD_IND
    7. SP_STATE_CODE
    8. BENE_COUNTY_CD
    9. BENE_HI_CVRAGE_TOT_MONS
   10. BENE_SMI_CVRAGE_TOT_MONS
   11. BENE_HMO_CVRAGE_TOT_MONS
   12. PLAN_CVRG_MOS_NUM
   13. SP_ALZHDMTA
   14. SP_CHF
   15. SP_CHRNKIDN
   16. SP_CNCR
   17. SP_COPD
   18. SP_DEPRESSN
   19. SP_DIABETES
   20. SP_ISCHMCHT
   21. SP_OSTEOPRS
   22. SP_RA_OA
   23. SP_STRKETIA
   24. MEDREIMB_IP
   25. BENRES_IP
   26. PPPYMT_IP
   27. MEDREIMB_OP
   28. BENRES_OP
   29. PPPYMT_OP
   30. MEDREIMB_CAR
   31. BENRES_CAR
   32. PPPYMT_CAR
   33. YEAR


In [12]:
# ---------------------------------------------------------
# Verify the critical columns we need for modeling are present.
# If any are missing, something is wrong with the file version.
# ---------------------------------------------------------

# These are the columns our model will depend on
REQUIRED_BENE_COLS = [
    "DESYNPUF_ID",      # Linking key across all files
    "BENE_BIRTH_DT",    # Used to calculate age at admission
    "BENE_SEX_IDENT_CD",# Sex
    "BENE_RACE_CD",     # Race (used for bias analysis)
    "SP_CHF",           # Congestive Heart Failure flag
    "SP_DIABETES",      # Diabetes flag
    "SP_COPD",          # COPD flag
    "SP_CHRNKIDN",      # Chronic Kidney Disease flag
    "SP_STRKETIA",      # Stroke / TIA flag
    "SP_ALZHDMTA",      # Alzheimer's flag
    "SP_DEPRESSN",      # Depression flag
    "SP_ISCHMCHT",      # Ischemic Heart Disease flag
    "SP_OSTEOPRS",      # Osteoporosis flag
    "SP_RA_OA",         # RA / Osteoarthritis flag
]

print("Checking for required beneficiary columns...\n")

missing_cols = [col for col in REQUIRED_BENE_COLS if col not in bene_all.columns]

if missing_cols:
    print(f"  ❌ MISSING COLUMNS: {missing_cols}")
    raise ValueError("Required columns not found. Check file version or filename.")
else:
    print("  ✅ All required beneficiary columns are present.")

Checking for required beneficiary columns...

  ✅ All required beneficiary columns are present.


## Section 4 — Load the Inpatient Claims File

The Inpatient Claims file contains one row per inpatient hospital claim, covering 2008–2010.
This is the primary file we use to:
- Identify index admissions (the initial hospital stay)
- Derive the readmission target variable (was the patient readmitted within 30 days?)
- Extract clinical features (length of stay, DRG code, diagnosis codes, etc.)

In [13]:
# ---------------------------------------------------------
# Load the Inpatient Claims file.
# This is a larger file than the beneficiary files because
# each beneficiary can have multiple inpatient claims.
# ---------------------------------------------------------

print("Loading Inpatient Claims (2008–2010)...")
print("(This may take 30–60 seconds depending on your machine)\n")

inpatient = pd.read_csv(
    RAW_DIR / FILES["inpatient"],
    low_memory=False
)

print(f"  Rows    : {inpatient.shape[0]:,}")
print(f"  Columns : {inpatient.shape[1]}")
print(f"  Memory  : {inpatient.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nFirst 3 rows of inpatient claims:")
inpatient.head(3)

Loading Inpatient Claims (2008–2010)...
(This may take 30–60 seconds depending on your machine)

  Rows    : 66,773
  Columns : 81
  Memory  : 88.7 MB

First 3 rows of inpatient claims:


,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,...,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00013D2EFD8E45D1,196661176988405,1,20100312.0,20100313.0,2600GD,4000.0,0.0,3.139084e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00016F745862898F,196201177000368,1,20090412.0,20090418.0,3900MB,26000.0,0.0,6.476809e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00016F745862898F,196661177015632,1,20090831.0,20090902.0,3900HM,5000.0,0.0,6.119985e+08,611998537.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# ---------------------------------------------------------
# Inspect all inpatient column names.
# The inpatient file has many ICD-9 diagnosis code columns
# (ICD9_DGNS_CD_1 through _10) which we will use
# to count diagnoses and group by condition category.
# ---------------------------------------------------------

print("All columns in the Inpatient Claims file:\n")

for i, col in enumerate(inpatient.columns):
    print(f"  {i+1:>3}. {col}")

All columns in the Inpatient Claims file:

    1. DESYNPUF_ID
    2. CLM_ID
    3. SEGMENT
    4. CLM_FROM_DT
    5. CLM_THRU_DT
    6. PRVDR_NUM
    7. CLM_PMT_AMT
    8. NCH_PRMRY_PYR_CLM_PD_AMT
    9. AT_PHYSN_NPI
   10. OP_PHYSN_NPI
   11. OT_PHYSN_NPI
   12. CLM_ADMSN_DT
   13. ADMTNG_ICD9_DGNS_CD
   14. CLM_PASS_THRU_PER_DIEM_AMT
   15. NCH_BENE_IP_DDCTBL_AMT
   16. NCH_BENE_PTA_COINSRNC_LBLTY_AM
   17. NCH_BENE_BLOOD_DDCTBL_LBLTY_AM
   18. CLM_UTLZTN_DAY_CNT
   19. NCH_BENE_DSCHRG_DT
   20. CLM_DRG_CD
   21. ICD9_DGNS_CD_1
   22. ICD9_DGNS_CD_2
   23. ICD9_DGNS_CD_3
   24. ICD9_DGNS_CD_4
   25. ICD9_DGNS_CD_5
   26. ICD9_DGNS_CD_6
   27. ICD9_DGNS_CD_7
   28. ICD9_DGNS_CD_8
   29. ICD9_DGNS_CD_9
   30. ICD9_DGNS_CD_10
   31. ICD9_PRCDR_CD_1
   32. ICD9_PRCDR_CD_2
   33. ICD9_PRCDR_CD_3
   34. ICD9_PRCDR_CD_4
   35. ICD9_PRCDR_CD_5
   36. ICD9_PRCDR_CD_6
   37. HCPCS_CD_1
   38. HCPCS_CD_2
   39. HCPCS_CD_3
   40. HCPCS_CD_4
   41. HCPCS_CD_5
   42. HCPCS_CD_6
   43. HCPCS_CD_7
 

In [16]:
# ---------------------------------------------------------
# Verify the critical inpatient columns are present.
# CLM_ADMSN_DT and NCH_BENE_DSCHRG_DT are essential —
# we cannot derive the readmission label without them.
# ---------------------------------------------------------

REQUIRED_IP_COLS = [
    "DESYNPUF_ID",           # Links back to the beneficiary file
    "CLM_ADMSN_DT",          # Admission date — defines the index event
    "NCH_BENE_DSCHRG_DT",   # Discharge date — start of 30-day window
    "CLM_DRG_CD",            # Diagnosis-Related Group code
    "CLM_PMT_AMT",           # Claim payment amount
    "CLM_UTLZTN_DAY_CNT",    # Total LOS
    "ICD9_DGNS_CD_1",        # Primary diagnosis code
]

print("Checking for required inpatient columns...\n")

missing_ip_cols = [col for col in REQUIRED_IP_COLS if col not in inpatient.columns]

if missing_ip_cols:
    print(f"  ❌ MISSING COLUMNS: {missing_ip_cols}")
    raise ValueError("Required inpatient columns not found. Check file version.")
else:
    print("  ✅ All required inpatient columns are present.")

Checking for required inpatient columns...

  ✅ All required inpatient columns are present.


## Section 5 — Basic Data Profiling

Before saving anything, we run a quick profile of both files.
This gives us a baseline understanding of the data quality
and surfaces any issues we need to handle in the feature engineering notebook.

In [17]:
# ---------------------------------------------------------
# Helper function: generate a simple null/dtype summary
# for any dataframe. We'll use this on both files.
# ---------------------------------------------------------

def profile_dataframe(df, name):
    """
    Print a concise profile of a dataframe:
    - Shape
    - Data types
    - Null counts and percentages for columns with any nulls
    - Duplicate row count
    """
    print(f"{'='*60}")
    print(f"PROFILE: {name}")
    print(f"{'='*60}")
    print(f"  Shape     : {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"  Duplicates: {df.duplicated().sum():,} exact duplicate rows")
    print()

    # Identify columns that have at least one null value
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]

    if len(cols_with_nulls) == 0:
        print("  ✅ No null values found in any column.")
    else:
        print(f"  Columns with null values ({len(cols_with_nulls)} of {df.shape[1]}):")
        print(f"  {'Column':<40} {'Null Count':>12} {'% Null':>10}")
        print(f"  {'-'*62}")
        for col, count in cols_with_nulls.items():
            pct = count / len(df) * 100
            print(f"  {col:<40} {count:>12,} {pct:>9.1f}%")

    print()

print("Profile function defined.")

Profile function defined.


In [18]:
# ---------------------------------------------------------
# Profile the combined beneficiary file.
# We expect ~2.3M rows across 3 years.
# Chronic condition flags (SP_*) should have no nulls —
# CMS defaults missing flags to 0 (not 1) in SynPUF.
# ---------------------------------------------------------

profile_dataframe(bene_all, "Beneficiary Summary (2008-2010 combined)")

PROFILE: Beneficiary Summary (2008-2010 combined)
  Shape     : 343,644 rows x 33 columns
  Duplicates: 0 exact duplicate rows

  Columns with null values (1 of 33):
  Column                                     Null Count     % Null
  --------------------------------------------------------------
  BENE_DEATH_DT                                 338,183      98.4%



In [19]:
# ---------------------------------------------------------
# Profile the inpatient claims file.
# We expect nulls in the secondary ICD-9 diagnosis columns
# (ICD9_DGNS_CD_2 through _10) since not every claim has
# 10 diagnoses. That is normal and expected.
# ---------------------------------------------------------

profile_dataframe(inpatient, "Inpatient Claims (2008-2010)")

PROFILE: Inpatient Claims (2008-2010)
  Shape     : 66,773 rows x 81 columns
  Duplicates: 0 exact duplicate rows

  Columns with null values (69 of 81):
  Column                                     Null Count     % Null
  --------------------------------------------------------------
  CLM_FROM_DT                                        68       0.1%
  CLM_THRU_DT                                        68       0.1%
  AT_PHYSN_NPI                                      673       1.0%
  OP_PHYSN_NPI                                   27,715      41.5%
  OT_PHYSN_NPI                                   59,090      88.5%
  ADMTNG_ICD9_DGNS_CD                               599       0.9%
  NCH_BENE_IP_DDCTBL_AMT                          2,178       3.3%
  CLM_UTLZTN_DAY_CNT                                 68       0.1%
  ICD9_DGNS_CD_1                                     95       0.1%
  ICD9_DGNS_CD_2                                    526       0.8%
  ICD9_DGNS_CD_3                            

In [20]:
# ---------------------------------------------------------
# Quick look at the beneficiary demographic distributions.
# These should roughly reflect the Medicare population:
#   Sex  : 1=Male, 2=Female — expect ~45% male, ~55% female
#   Race : 1=White, 2=Black, 3=Other, 4=Asian, 5=Hispanic, 6=North American Native
# ---------------------------------------------------------

print("Beneficiary Sex Distribution (2008 only):")
sex_dist = bene_2008["BENE_SEX_IDENT_CD"].value_counts(normalize=True) * 100
print(sex_dist.to_string())

print("\nBeneficiary Race Distribution (2008 only):")
race_dist = bene_2008["BENE_RACE_CD"].value_counts(normalize=True) * 100
print(race_dist.to_string())

print("\nChronic Condition Prevalence (2008, % of beneficiaries with flag = 1):")
condition_cols = [c for c in bene_2008.columns if c.startswith("SP_")]
condition_rates = (bene_2008[condition_cols] == 1).mean() * 100
print(condition_rates.sort_values(ascending=False).to_string())

Beneficiary Sex Distribution (2008 only):
BENE_SEX_IDENT_CD
2    55.303733
1    44.696267

Beneficiary Race Distribution (2008 only):
BENE_RACE_CD
1    82.808203
2    10.608326
3     4.238002
5     2.345469

Chronic Condition Prevalence (2008, % of beneficiaries with flag = 1):
SP_ISCHMCHT      42.063738
SP_DIABETES      37.867849
SP_CHF           28.495428
SP_DEPRESSN      21.349010
SP_ALZHDMTA      19.260520
SP_OSTEOPRS      17.341344
SP_CHRNKIDN      16.059887
SP_RA_OA         15.398102
SP_COPD          13.530494
SP_CNCR           6.372903
SP_STRKETIA       4.488965
SP_STATE_CODE     2.208815


In [21]:
# ---------------------------------------------------------
# Quick look at inpatient claims date ranges.
# Admission dates should span 2008-2010.
# We also check for any obviously bad dates (e.g., nulls,
# discharge before admission).
# ---------------------------------------------------------

# Convert date columns from integer format (YYYYMMDD) to proper dates
# SynPUF dates are stored as integers — e.g., 20080115 = Jan 15, 2008
inpatient["CLM_ADMSN_DT"]        = pd.to_datetime(inpatient["CLM_ADMSN_DT"],        format="%Y%m%d", errors="coerce")
inpatient["NCH_BENE_DSCHRG_DT"]  = pd.to_datetime(inpatient["NCH_BENE_DSCHRG_DT"],  format="%Y%m%d", errors="coerce")

print("Inpatient Admission Date Range:")
print(f"  Earliest admission : {inpatient['CLM_ADMSN_DT'].min()}")
print(f"  Latest admission   : {inpatient['CLM_ADMSN_DT'].max()}")
print(f"  Null admission dates: {inpatient['CLM_ADMSN_DT'].isnull().sum():,}")

print("\nInpatient Discharge Date Range:")
print(f"  Earliest discharge : {inpatient['NCH_BENE_DSCHRG_DT'].min()}")
print(f"  Latest discharge   : {inpatient['NCH_BENE_DSCHRG_DT'].max()}")
print(f"  Null discharge dates: {inpatient['NCH_BENE_DSCHRG_DT'].isnull().sum():,}")

# Flag any records where discharge is before admission — these are data errors
bad_dates = inpatient[inpatient["NCH_BENE_DSCHRG_DT"] < inpatient["CLM_ADMSN_DT"]]
print(f"\nClaims where discharge < admission (data errors): {len(bad_dates):,}")

Inpatient Admission Date Range:
  Earliest admission : 2007-11-27 00:00:00
  Latest admission   : 2010-12-30 00:00:00
  Null admission dates: 0

Inpatient Discharge Date Range:
  Earliest discharge : 2008-01-01 00:00:00
  Latest discharge   : 2010-12-31 00:00:00
  Null discharge dates: 0

Claims where discharge < admission (data errors): 0


In [22]:
# ---------------------------------------------------------
# Check how many unique beneficiaries appear in each file.
# Both files should share the same DESYNPUF_ID universe —
# this confirms the files are from the same Sample 1.
# ---------------------------------------------------------

bene_ids      = set(bene_2008["DESYNPUF_ID"].unique())
inpatient_ids = set(inpatient["DESYNPUF_ID"].unique())

print(f"Unique beneficiaries in beneficiary file (2008) : {len(bene_ids):,}")
print(f"Unique beneficiaries in inpatient file          : {len(inpatient_ids):,}")

# Beneficiaries who had at least one inpatient stay
overlap = bene_ids.intersection(inpatient_ids)
print(f"Beneficiaries appearing in BOTH files           : {len(overlap):,}")
print(f"  (This is the linkable population for modeling)")

# Beneficiaries in inpatient but NOT in beneficiary — should be zero
ip_only = inpatient_ids - bene_ids
print(f"\nBeneficiaries in inpatient but not in bene file : {len(ip_only):,}")
if len(ip_only) > 0:
    print("  ⚠️  Warning: some inpatient IDs have no beneficiary record.")
    print("  This can happen in SynPUF due to synthetic data generation.")
    print("  These records will be excluded when we join in notebook 03.")

Unique beneficiaries in beneficiary file (2008) : 116,352
Unique beneficiaries in inpatient file          : 37,780
Beneficiaries appearing in BOTH files           : 37,780
  (This is the linkable population for modeling)

Beneficiaries in inpatient but not in bene file : 0


## Section 6 — Save Outputs

Save two types of output files, both prefixed with `01_` to match this notebook:
1. **Preview CSVs** — first 1,000 rows of each file, for quick reference in downstream notebooks without reloading the full file.
2. **Validation report** — a text summary of everything we checked above, useful for the README and model card.

In [23]:
# ---------------------------------------------------------
# Save preview files — first 1000 rows of each dataset.
# Prefix 01_ ties them back to this notebook.
# These are small enough to open instantly in any editor.
# ---------------------------------------------------------

# Beneficiary preview (2008 only — our primary year)
bene_preview_path = PROCESSED_DIR / "01_beneficiary_2008_preview.csv"
bene_2008.head(1000).to_csv(bene_preview_path, index=False)
print(f"Saved: {bene_preview_path}")

# Inpatient preview
ip_preview_path = PROCESSED_DIR / "01_inpatient_preview.csv"
inpatient.head(1000).to_csv(ip_preview_path, index=False)
print(f"Saved: {ip_preview_path}")

Saved: /workspaces/readmitiq/data/processed/01_beneficiary_2008_preview.csv
Saved: /workspaces/readmitiq/data/processed/01_inpatient_preview.csv


In [24]:
# ---------------------------------------------------------
# Save a text-based validation report.
# This documents what we loaded and what we found.
# Useful for the README, model card, and debugging later.
# ---------------------------------------------------------

report_path = PROCESSED_DIR / "01_validation_report.txt"

with open(report_path, "w") as f:
    f.write("ReadmitIQ — Data Validation Report\n")
    f.write(f"Generated  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Source     : CMS DE-SynPUF Sample 1 (2008-2010)\n")
    f.write("=" * 60 + "\n\n")

    f.write("BENEFICIARY SUMMARY FILE\n")
    f.write("-" * 40 + "\n")
    f.write(f"  2008 rows            : {bene_2008.shape[0]:,}\n")
    f.write(f"  2009 rows            : {bene_2009.shape[0]:,}\n")
    f.write(f"  2010 rows            : {bene_2010.shape[0]:,}\n")
    f.write(f"  Combined rows        : {bene_all.shape[0]:,}\n")
    f.write(f"  Columns              : {bene_all.shape[1]}\n")
    f.write(f"  Unique bene IDs 2008 : {bene_2008['DESYNPUF_ID'].nunique():,}\n\n")

    f.write("INPATIENT CLAIMS FILE\n")
    f.write("-" * 40 + "\n")
    f.write(f"  Total rows           : {inpatient.shape[0]:,}\n")
    f.write(f"  Columns              : {inpatient.shape[1]}\n")
    f.write(f"  Unique bene IDs      : {inpatient['DESYNPUF_ID'].nunique():,}\n")
    f.write(f"  Admission date range : {inpatient['CLM_ADMSN_DT'].min()} to {inpatient['CLM_ADMSN_DT'].max()}\n")
    f.write(f"  Bad date records     : {len(bad_dates):,}\n")
    f.write(f"  Linkable bene IDs    : {len(overlap):,}\n\n")

    f.write("REQUIRED COLUMN CHECKS\n")
    f.write("-" * 40 + "\n")
    f.write("  Beneficiary required columns : ALL PRESENT\n")
    f.write("  Inpatient required columns   : ALL PRESENT\n")

print(f"Saved: {report_path}")

Saved: /workspaces/readmitiq/data/processed/01_validation_report.txt


## Section 7 — Summary and Next Steps

In [25]:
# ---------------------------------------------------------
# Final summary — print a clean status report so you know
# exactly what was loaded and what is ready for the next notebook.
# ---------------------------------------------------------

print("=" * 60)
print("NOTEBOOK 01 — COMPLETE")
print("=" * 60)
print()
print("What was loaded:")
print(f"  Beneficiary Summary (all years) : {bene_all.shape[0]:,} rows")
print(f"  Inpatient Claims                : {inpatient.shape[0]:,} rows")
print(f"  Linkable population             : {len(overlap):,} unique beneficiaries")
print()
print("Files saved to data/processed/:")
print("  01_beneficiary_2008_preview.csv")
print("  01_inpatient_preview.csv")
print("  01_validation_report.txt")
print()
print("Next step:")
print("  Open 02_eda.ipynb to explore the data visually before")
print("  building features in 03_feature_engineering.ipynb.")
print("=" * 60)

NOTEBOOK 01 — COMPLETE

What was loaded:
  Beneficiary Summary (all years) : 343,644 rows
  Inpatient Claims                : 66,773 rows
  Linkable population             : 37,780 unique beneficiaries

Files saved to data/processed/:
  01_beneficiary_2008_preview.csv
  01_inpatient_preview.csv
  01_validation_report.txt

Next step:
  Open 02_eda.ipynb to explore the data visually before
  building features in 03_feature_engineering.ipynb.


# Summary
- 343,644 beneficiary rows across 3 years = ~114K unique beneficiaries per year. Sample 1 is 1 of 20 subsamples, so this is working as expected.
- 66,773 inpatient claims across 2008–2010 = roughly 22K admissions per year. That's what we'll split into train (2008) and test (2009).
- 37,780 linkable beneficiaries — meaning about 33% of beneficiaries had at least one inpatient stay over the 3 years, which is realistic for a Medicare population.